# 03. Open Manhole: YOLOv8 detection on the Road Hazards Dataset

The Road Hazards Dataset labels three classes: `0` pothole, `1` crack, `2` open manhole. We keep only the open manhole boxes (renumbered to class 0) and train `open_manhole_model.pt`.

Training split: all 673 images with a manhole plus 450 randomly chosen images without one (seed 0), so the model also learns what is not a manhole. Validation: the dataset's full validation split (480 images, 148 with manholes).

Our run: stopped early at epoch 42, about 34 minutes. Validation box mAP50 0.908, precision 0.890, recall 0.874. The dataset contains augmented copies of some photos, so this score may be optimistic. Run `00_download_datasets.ipynb` first.

In [ ]:
from pathlib import Path

# Works whether the notebook is opened from ai_model/ or ai_model/notebooks/.
ROOT = Path.cwd().resolve()
while ROOT.name != "ai_model" and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert ROOT.name == "ai_model", "Open this notebook from inside the ai_model folder"

DATA = ROOT / "data"          # downloaded and converted datasets (git-ignored)
RAW = DATA / "raw"            # original downloads
RUNS = ROOT / "runs"          # Ultralytics training runs (git-ignored)
WEIGHTS = ROOT / "weights"    # final model files used by the AI service
RESULTS = ROOT / "results"    # metrics used in the project report
for p in (DATA, RAW, RUNS, WEIGHTS, RESULTS):
    p.mkdir(parents=True, exist_ok=True)
print("ai_model root:", ROOT)

In [ ]:
import torch

# Apple Silicon uses the MPS backend; an NVIDIA GPU uses CUDA; otherwise CPU.
DEVICE = 0 if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("training device:", DEVICE)

In [ ]:
import random, shutil

NAME = "open_manhole"
src = RAW / "road_hazards" / "dataset" / "dataset"
out = DATA / NAME
data_yaml = out / "data.yaml"
MANHOLE = "2"

if not (out / "images" / "train").exists():
    random.seed(0)
    for split, sname, n_neg in (("train", "train", 450), ("val", "valid", 10**6)):
        (out / "images" / split).mkdir(parents=True, exist_ok=True)
        (out / "labels" / split).mkdir(parents=True, exist_ok=True)
        pos, neg = [], []
        for lf in sorted((src / sname / "labels").glob("*.txt")):
            rows = [l.split() for l in open(lf) if len(l.split()) >= 5]
            keep = [r for r in rows if r[0] == MANHOLE]
            (pos if keep else neg).append((lf, keep))
        random.shuffle(neg)
        n = 0
        for lf, keep in pos + neg[:n_neg]:
            img = src / sname / "images" / f"{lf.stem}.jpg"
            if not img.exists():
                continue
            shutil.copy(img, out / "images" / split / img.name)
            (out / "labels" / split / lf.name).write_text("".join("0 " + " ".join(r[1:5]) + "\n" for r in keep))
            n += 1
        print(split, n, "images,", len(pos), "with a manhole")

data_yaml.write_text(f"path: {out}\ntrain: images/train\nval: images/val\nnames: ['Open manhole']\n")

In [ ]:
import shutil
from ultralytics import YOLO


def train(name, data_yaml, task, epochs):
    """Fine-tune a COCO-pretrained YOLOv8 nano model with the settings used for every UrbanFix category."""
    base = "yolov8n-seg.pt" if task == "segment" else "yolov8n.pt"
    model = YOLO(base)
    model.train(
        data=str(data_yaml),
        epochs=epochs,
        imgsz=640,
        batch=16,
        patience=15,          # stop if validation fitness does not improve for 15 epochs
        device=DEVICE,
        seed=0,
        deterministic=False,
        workers=4,
        plots=True,
        project=str(RUNS),
        name=name,
        exist_ok=True,
        verbose=False,
    )
    best = RUNS / name / "weights" / "best.pt"
    target = WEIGHTS / f"{name}_model.pt"
    shutil.copy(best, target)
    print("saved", target)
    return target

In [ ]:
train(NAME, data_yaml, task="detect", epochs=45)

In [ ]:
def quick_val(name, data_yaml):
    """Validate the saved model and print the headline numbers."""
    m = YOLO(str(WEIGHTS / f"{name}_model.pt"))
    r = m.val(data=str(data_yaml), imgsz=640, batch=16, device=DEVICE, plots=False, verbose=False,
              project=str(RUNS / "val"), name=name, exist_ok=True)
    print(f"box  P {r.box.mp:.3f}  R {r.box.mr:.3f}  mAP50 {r.box.map50:.3f}  mAP50-95 {r.box.map:.3f}")
    if hasattr(r, "seg") and r.seg is not None and m.task == "segment":
        print(f"mask P {r.seg.mp:.3f}  R {r.seg.mr:.3f}  mAP50 {r.seg.map50:.3f}  mAP50-95 {r.seg.map:.3f}")
    return r

In [ ]:
quick_val(NAME, data_yaml);